##### Copyright 2026 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Generative quantum advantage

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/quantum/tutorials/generative_quantum_advantage"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/quantum/blob/master/docs/tutorials/generative_quantum_advantage.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/quantum/blob/master/docs/tutorials/generative_quantum_advantage.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/quantum/docs/tutorials/generative_quantum_advantage.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

This tutorial introduces <a href="https://arxiv.org/abs/2509.09033" class="external">generative quantum advantage</a> from Huang, Chen, and collaborators. The paper shows quantum models that can be trained efficiently from classical data, yet generate samples from distributions that are hard to simulate classically at scale.

You will not reproduce the full device scale experiment here. Instead you will build intuition for the core ideas and run a small end to end demo with TensorFlow Quantum (TFQ):

1. Define a shallow parameterized quantum circuit in the spirit of instantaneously deep quantum neural networks (IDQNNs).
2. Create classical training data from a fixed target distribution.
3. Train the circuit parameters with TFQ.
4. Sample bitstrings from the trained model and compare with the target.

Unlike the [Quantum data](https://www.tensorflow.org/quantum/tutorials/quantum_data) tutorial, which focuses on discriminative learning and dataset characterization, this notebook is **generative**: fit a quantum distribution and draw new samples from it. The training step uses the parameter shift rule from the [Calculate gradients](https://www.tensorflow.org/quantum/tutorials/gradients) tutorial. For a different quantum advantage setting built around convolutional structure, see [Quantum CNN](https://www.tensorflow.org/quantum/tutorials/qcnn).

## Setup

Install TensorFlow and TensorFlow Quantum:

In [ ]:
# In Colab, you will be asked to restart the session after this finishes.
!pip install tensorflow==2.18.1 tensorflow-quantum==0.7.6

Configure the use of Keras 2:

In [ ]:
import os

# Keras 2 must be selected before importing TensorFlow or TensorFlow Quantum:
os.environ["TF_USE_LEGACY_KERAS"] = "1"

Now import TensorFlow, TensorFlow Quantum, and other modules needed:

In [ ]:
import collections

import cirq
import numpy as np
import sympy
import tensorflow as tf
import tensorflow_quantum as tfq

%matplotlib inline
import matplotlib.pyplot as plt
from cirq.contrib.svg import SVGCircuit

np.random.seed(42)
tf.random.set_seed(42)

## 1. Generative quantum advantage in brief

A generative problem asks a model to produce new outputs that match a target pattern. In the paper, the pattern can be classical bitstrings drawn from a conditional distribution $p(y|x)$, or compressed quantum circuits that speed up simulation.

Generative quantum advantage means a quantum computer can learn and generate those outputs substantially better than any classical computer. A key theme in Huang et al. is a practical split:

* **Training** can be efficient with classical optimization on modest data.
* **Inference / sampling** from the learned quantum model can require a quantum computer once the model is large enough.

This differs from earlier beyond classical sampling demonstrations where learning the model itself was not efficient. The paper closes that gap with shallow families such as IDQNNs that train well and remain hard to sample from classically at scale.

The demo below uses an unconditional bitstring distribution. The same train then sample workflow extends to the conditional case $p(y|x)$ described in the paper, where classical inputs select which quantum distribution to generate.

## 2. Shallow circuits and IDQNNs

IDQNNs (instantaneously deep quantum neural networks) are shallow quantum circuits that can be trained efficiently, yet sample from distributions that correspond to much deeper quantum processes. Intuitively, a shallow circuit prepares a state from which measurement outcomes are classically hard to reproduce when the system is large enough and sufficiently entangled.

In this tutorial you use a simple brickwork ansatz with fixed depth on a line of qubits. It is not a full IDQNN construction from the appendix, but it captures the same workflow the paper emphasizes:

1. Keep depth constant.
2. Learn parameters from classical bitstring samples.
3. Use quantum sampling to generate new data.

The paper proves and demonstrates much stronger statements at tens to thousands of qubits. Here you work with eight qubits so everything runs quickly in simulation while still showing the train then sample pattern.

## 3. Build a shallow generative ansatz

The following function builds a depth $d$ brickwork circuit: single qubit $R_x$ rotations followed by nearest neighbor $CZ$ gates. The same structure is used for the fixed target distribution and for the trainable student model.

In [ ]:
def build_shallow_ansatz(qubits, depth, param_prefix):
    """Return a brickwork parameterized circuit and its symbols."""
    symbols = []
    circuit = cirq.Circuit()
    for layer in range(depth):
        for idx, qubit in enumerate(qubits):
            symbol = sympy.Symbol(f'{param_prefix}_{layer}_{idx}')
            symbols.append(symbol)
            circuit.append(cirq.rx(symbol)(qubit))
        for idx in range(len(qubits) - 1):
            circuit.append(cirq.CZ(qubits[idx], qubits[idx + 1]))
    return circuit, symbols


n_qubits = 8
depth = 3
qubits = cirq.LineQubit.range(n_qubits)

target_circuit, target_symbols = build_shallow_ansatz(qubits, depth, 'target')
student_circuit, student_symbols = build_shallow_ansatz(qubits, depth, 'theta')

SVGCircuit(target_circuit[:6])

## 4. Create classical training data

Pick random parameters for the target circuit and treat the resulting distribution as the unknown generative problem. Sample bitstrings classically from that distribution. These samples are the training data for the student model.

This mirrors the paper setting where classical bitstrings are available during training even though generation is quantum.

In [ ]:
def bitstring_probabilities(circuit, symbols, param_values, qubits):
    """Exact computational basis probabilities for a pure state circuit."""
    resolver = cirq.ParamResolver(
        {symbol: value for symbol, value in zip(symbols, param_values)})
    state = cirq.Simulator().simulate(
        circuit, param_resolver=resolver).final_state_vector
    n = len(qubits)
    return {
        format(index, f'0{n}b'): float(np.abs(state[index])**2)
        for index in range(2**n)
    }


def sample_bitstrings(probabilities, n_samples):
    labels = list(probabilities.keys())
    probs = np.array([probabilities[label] for label in labels])
    probs = probs / np.sum(probs)
    indices = np.random.choice(len(labels), size=n_samples, p=probs)
    return [labels[index] for index in indices]


NUM_TRAINING_SAMPLES = 256

target_params = np.random.uniform(0.0, 2 * np.pi, size=len(target_symbols))
target_probabilities = bitstring_probabilities(target_circuit, target_symbols,
                                             target_params, qubits)
training_bitstrings = sample_bitstrings(target_probabilities,
                                        n_samples=NUM_TRAINING_SAMPLES)

print('Example training bitstrings:', training_bitstrings[:5])

## 5. Train the student circuit with TFQ

The paper trains shallow quantum models from classical bitstring samples by maximizing likelihood. You follow the same objective here with a negative log likelihood loss on the training bitstrings.

Use `tfq.get_state_op()` to evaluate amplitudes from the student circuit. Because the state op is not autodiff-friendly, optimize with the parameter shift rule for $R_x$ rotations. Apply the shift to bitstring **probabilities**, then combine with the log term through the chain rule. That matches the gradient recipe in the [Calculate gradients](https://www.tensorflow.org/quantum/tutorials/gradients) tutorial.

In [ ]:
NUM_TRAINING_STEPS = 80
LEARNING_RATE = 0.2
PARAMETER_SHIFT = np.pi / 2

symbol_names = [str(symbol) for symbol in student_symbols]
circuit_tensor = tfq.convert_to_tensor([student_circuit])
state_op = tfq.get_state_op()

bitstring_indices = np.array(
    [int(bitstring, 2) for bitstring in training_bitstrings])

trainable_params = np.random.uniform(
    0.0, 2 * np.pi, size=len(student_symbols)).astype(np.float32)


def get_bitstring_probabilities(param_vector):
    symbol_values = tf.expand_dims(
        tf.convert_to_tensor(param_vector, dtype=tf.float32), axis=0)
    state = state_op(circuit_tensor, symbol_names, symbol_values)
    amplitudes = state[0]
    probabilities = tf.math.real(amplitudes * tf.math.conj(amplitudes))
    return tf.gather(probabilities, bitstring_indices)


def evaluate_negative_log_likelihood(param_vector):
    selected = get_bitstring_probabilities(param_vector)
    return float(-tf.reduce_mean(tf.math.log(selected + 1e-9)).numpy())


def parameter_shift_gradients(param_vector, shift=PARAMETER_SHIFT):
    gradients = np.zeros_like(param_vector)
    prob_current = get_bitstring_probabilities(param_vector).numpy()
    for index in range(len(param_vector)):
        plus = param_vector.copy()
        minus = param_vector.copy()
        plus[index] += shift
        minus[index] -= shift
        prob_plus = get_bitstring_probabilities(plus).numpy()
        prob_minus = get_bitstring_probabilities(minus).numpy()
        gradients[index] = -np.mean(
            (prob_plus - prob_minus) / (2.0 * prob_current + 1e-9))
    return gradients


loss_history = []

for step in range(NUM_TRAINING_STEPS):
    loss = evaluate_negative_log_likelihood(trainable_params)
    gradients = parameter_shift_gradients(trainable_params)
    trainable_params -= LEARNING_RATE * gradients
    loss_history.append(loss)
    if step % 20 == 0:
        print(f'step {step:02d}  nll {loss_history[-1]:.4f}')

plt.plot(loss_history)
plt.xlabel('training step')
plt.ylabel('negative log likelihood')
plt.title('Training a shallow generative circuit')
plt.show()

## 6. Sample from the trained model

After training, use the TFQ sampling op to draw bitstrings from the student circuit. Compare the sampled histogram with the target distribution.

In [ ]:
NUM_SAMPLE_DRAWS = 1024

measurement_circuit = student_circuit + cirq.measure(*qubits, key='m')
measurement_tensor = tfq.convert_to_tensor([measurement_circuit])
sample_op = tfq.get_sampling_op()

symbol_values = tf.expand_dims(
    tf.convert_to_tensor(trainable_params, dtype=tf.float32), axis=0)
num_samples = tf.constant([NUM_SAMPLE_DRAWS], dtype=tf.int32)
samples = sample_op(measurement_tensor, symbol_names, symbol_values,
                    num_samples)
sample_array = samples.numpy()[0]

student_histogram = collections.Counter(
    ''.join(str(int(bit)) for bit in sample) for sample in sample_array)

student_probabilities = bitstring_probabilities(student_circuit,
                                              student_symbols,
                                              trainable_params,
                                              qubits)


def hellinger_fidelity(distribution_a, distribution_b):
    shared_keys = distribution_a.keys() & distribution_b.keys()
    return sum(
        np.sqrt(distribution_a[key] * distribution_b[key])
        for key in shared_keys)


overlap = hellinger_fidelity(target_probabilities, student_probabilities)
print(f'Hellinger fidelity between target and trained model: {overlap:.4f}')

top_keys = sorted(
    target_probabilities, key=target_probabilities.get, reverse=True)[:8]
x_labels = top_keys
target_vals = [target_probabilities[key] for key in x_labels]
student_vals = [
    student_histogram.get(key, 0) / NUM_SAMPLE_DRAWS for key in x_labels
]

positions = np.arange(len(x_labels))
width = 0.35
plt.bar(positions - width / 2, target_vals, width, label='target')
plt.bar(positions + width / 2, student_vals, width, label='student samples')
plt.xticks(positions, x_labels, rotation=45)
plt.ylabel('probability')
plt.title('Target vs trained distribution (top bitstrings)')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Where classical hardness enters

With eight qubits you can compute amplitudes and probabilities exactly on a laptop. That is intentional. The pedagogical point is the workflow: classical data, efficient training of a shallow quantum model, and quantum sampling for generation.

Huang et al. scale this idea to tens and thousands of qubits and connect shallow IDQNNs to distributions that are beyond classical polynomial time sampling under standard complexity assumptions. The hardness is not in training on classical samples. It is in generating new samples from the learned quantum model without a quantum computer.

The paper also studies a second generative task, learning compressed simulation circuits with the sewing technique. That extension is a natural follow up but is outside the scope of this notebook.

## 8. Takeaways

In this tutorial you:

* Built a shallow parameterized circuit in TFQ.
* Generated classical training bitstrings from a target quantum distribution.
* Trained circuit parameters with `tfq.get_state_op()` and the parameter shift rule.
* Sampled from the trained model with `tfq.get_sampling_op()` and compared output statistics with the target.

These steps mirror the generative quantum advantage story in <a href="https://arxiv.org/abs/2509.09033" class="external">Huang et al.</a> at tutorial scale. For the full experimental results, hardness theorems, IDQNN constructions, and the circuit sewing extension, see the paper and its appendices.